In [1]:
import pandas as pd

df = pd.read_csv("../data/CRMLSSold_merge.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_4483/3104117343.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/CRMLSSold_merge.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,month,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,202401,NaN,NaN,NaN,NaN
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,NaN,52320.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,NaN,217364.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,NaN,217800.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,0.0,108883.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN


In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", len(cat_cols))
print(cat_cols)

Categorical columns: 53
Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'ListAgentEmail', 'CloseDate',
       'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress',
       'PropertyType', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish',
       'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'BuilderName',
       'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR', 'ListingId',
       'City', 'ContractStatusChangeDate', 'CoBuyerAgentFirstName',
       'PurchaseContractDate', 'ListingContractDate', 'BusinessType',
       'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool',
       'Levels', 'LotSizeDimensions', 'NewConstructionYN',
       'HighSchoolDistrict', 'PostalCod

In [4]:
import pandas as pd

# =========================
# 1. Identify categorical columns
# =========================
cat_cols = df.select_dtypes(include=["object", "string"]).columns

# =========================
# 2. Build base review table
# =========================
cat_review = pd.DataFrame({
    "column": cat_cols,
    "missing_count": df[cat_cols].isnull().sum().values,
    "missing_pct": (df[cat_cols].isnull().mean().values * 100).round(2),
    "n_unique": df[cat_cols].nunique(dropna=True).values
})

# =========================
# 3. Sample top values (for quick inspection)
# =========================
top_values = []
for col in cat_cols:
    vals = df[col].value_counts(dropna=False).head(5)
    vals_str = "; ".join([f"{str(idx)} ({cnt})" for idx, cnt in vals.items()])
    top_values.append(vals_str)

cat_review["top_values_sample"] = top_values

# =========================
# 4. Define core categorical columns to keep
# =========================
core_columns = [
    "PropertyType",
    "PropertySubType",
    "City",
    "PostalCode",
    "CountyOrParish",
    "StateOrProvince",
    "MLSAreaMajor",
    "MlsStatus",
    "CloseDate",
    "ContractStatusChangeDate",
    "PurchaseContractDate"
]

# =========================
# 5. Decision logic
# =========================
def classify_column(row):
    col = row["column"]
    missing_pct = row["missing_pct"]
    
    if col in core_columns:
        return "Keep (Core)"
    elif missing_pct >= 90:
        return "Drop"
    else:
        return "Review"


cat_review["decision"] = cat_review.apply(classify_column, axis=1)
# =========================
# 6. Sort by missing percentage
# =========================
cat_review = cat_review.sort_values(by="missing_pct", ascending=False)

# =========================
# 7. Display results
# =========================
print("=== KEEP (CORE) ===")
display(cat_review[cat_review["decision"] == "Keep (Core)"])

print("=== DROP ===")
display(cat_review[cat_review["decision"] == "Drop"])

print("=== Review ===")
display(cat_review[cat_review["decision"] == "Review"])



=== KEEP (CORE) ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
23,MLSAreaMajor,65001,11.00,1126,nan (65001); 699 - Not Defined (54335); SRCAR ...,Keep (Core)
29,PropertySubType,45318,7.67,34,SingleFamilyResidence (360943); Condominium (1...,Keep (Core)
36,PurchaseContractDate,11824,2.00,1237,nan (11824); 2024-07-01 (1287); 2024-05-01 (12...,Keep (Core)
34,ContractStatusChangeDate,712,0.12,821,2024-06-28 (1898); 2024-03-29 (1748); 2026-02-...,Keep (Core)
33,City,440,0.07,1237,Los Angeles (39595); San Diego (23192); Irvine...,Keep (Core)
47,PostalCode,165,0.03,4145,92618 (3458); 92253 (3042); 92677 (2659); 9221...,Keep (Core)
12,PropertyType,0,0.00,8,Residential (397192); ResidentialLease (135456...,Keep (Core)
24,CountyOrParish,0,0.00,63,Los Angeles (179062); Orange (79157); Riversid...,Keep (Core)
25,MlsStatus,0,0.00,1,Closed (591071),Keep (Core)
39,StateOrProvince,0,0.00,25,CA (590994); AZ (16); NV (8); OK (7); TX (6),Keep (Core)


=== DROP ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
4,WaterfrontYN,590749,99.95,1,nan (590749); True (322),Drop
38,BusinessType,589439,99.72,760,nan (589439); Restaurant (166); Retail (74); O...,Drop
5,BasementYN,581333,98.35,1,nan (581333); True (9738),Drop
28,BuilderName,568496,96.18,3935,nan (568496); Lennar (1998); Richmond American...,Drop
44,LotSizeDimensions,560061,94.75,13078,nan (560061); 50x135 (584); 50x150 (449); 50x1...,Drop
35,CoBuyerAgentFirstName,543074,91.88,4841,nan (543074); Michael (689); David (559); Andr...,Drop


=== Review ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
49,OriginatingSystemSubName,531106,89.85,10,nan (531106); CRMLS_CRM (34254); CRMLS_CL (682...,Review
48,OriginatingSystemName,531106,89.85,1,nan (531106); CRMLS (59965),Review
26,ElementarySchool,525488,88.90,1419,nan (525488); Other (10219); Harbor View (558)...,Review
40,MiddleOrJuniorSchool,525047,88.83,669,nan (525047); Other (7690); Corona Del Mar (94...,Review
50,BuyerAgencyCompensationType,523324,88.54,3,nan (523324); Item1 (61533); Item (6071); SeeR...,Review
42,HighSchool,505027,85.44,525,nan (505027); Other (3016); Tesoro (1458); Dan...,Review
51,latfilled,495278,83.79,2,nan (495278); False (50537); True (45256),Review
52,lonfilled,495278,83.79,2,nan (495278); False (50639); True (45154),Review
17,CoListAgentFirstName,468885,79.33,7347,nan (468885); Michael (1890); David (1681); Jo...,Review
18,CoListAgentLastName,468633,79.29,15392,nan (468633); Myatt (890); Smith (716); Lee (5...,Review


##Second Round of Screening: Review Phase

Drop: OriginatingSystemSubName, OriginatingSystemName,
    CoListAgentFirstName, CoListAgentLastName, BuyerAgentAOR,
    ListAgentAOR, ListAgentEmail, BuyerOfficeAOR, ListAgentFirstName, 
    BuyerAgentFirstName, BuyerAgentLastName, ListAgentFullName

keep: 
ElementarySchool: -School attendance information for the assigned elementary school.

MiddleOrJuniorSchool: -School attendance information for the assigned middle or junior school.

BuyerAgencyCompensationType:- Type of compensation offered to the buyer’s agent.

HighSchool: -School attendance information for the assigned high school.

latfilled: -Filled or imputed latitude value for the property location.

lonfilled: -Filled or imputed longitude value for the property location.

AssociationFeeFrequency: -Frequency of HOA fee payments, such as monthly or yearly.

Flooring: -List or description of flooring materials used in the property.

HighSchoolDistrict: -High school district information associated with the property.

AttachedGarageYN: -Indicates whether the property has an attached garage (Yes/No).

Levels: -Number of levels or floors in the property.

PoolPrivateYN: -Indicates whether the property has a private pool.

NewConstructionYN: -Indicates whether the property is newly constructed.

ViewYN: -Indicates whether the property has a notable view (e.g., ocean, city).

FireplaceYN: -Indicates whether the property has a fireplace.


BuyerOfficeName: Name of the buyer’s brokerage office.

BuyerAgentMlsId: Internal MLS identifier for the buyer’s agent.

UnparsedAddress: Full unstructured property address.

ListingContractDate: Date when the property was listed on the market.

ListOfficeName: Name of the listing (seller’s) brokerage office.

ListingId: Unique identifier for the property listing, used for tracking or display.
